# Reading Vector Patterns (RVP) for Near-Duplicate Short Text Detection

This notebook demonstrates the RVP method for detecting near-duplicate short text snippets. 

## What This Demo Does
- Implements Reading Vector Patterns using cognitive load indicators (word length, frequency, POS tags)
- Uses Dynamic Time Warping (DTW) to compare sequences of reading vectors
- Compares RVP against baseline methods: Jaccard 3-gram, MinHash LSH, and Sentence-BERT
- Evaluates on a small demo dataset with precision, recall, and F1-score

## Dataset
This demo uses a curated subset of 20 question pairs from the Quora Question Pairs dataset.

## Expected Runtime
With the mini demo data (20 examples), this notebook should run in under 2 minutes.

In [ ]:
# Install dependencies - follows aii-colab pattern for Colab compatibility
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('loguru')
_pip('datasketch')
_pip('sentence-transformers')
_pip('fastdtw')
_pip('gensim')
_pip('tqdm')
_pip('nltk')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

print('Dependencies installed successfully!')

In [ ]:
# Standard imports - copied from original method.py
import json
import sys
import os
import time
import random
import tracemalloc
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass
from loguru import logger
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
from scipy.stats import bootstrap
import nltk
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
import re
from collections import Counter
from datasketch import MinHash, MinHashLSH
from sentence_transformers import SentenceTransformer
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import gensim
from tqdm import tqdm
import matplotlib.pyplot as plt

# Download NLTK data if needed
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger', quiet=True)

print('All imports successful!')

In [ ]:
# Data loading helper - uses GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-c84097-reading-vector-patterns-rvp-a-cognitive/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL or local file."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub URL failed: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        print("Loading from local file...")
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

print('Data loading helper defined!')

In [ ]:
# Load the demo data
print('Loading demo data...')
data = load_data()

# Extract examples from the loaded data
all_examples = []
if "datasets" in data and isinstance(data["datasets"], list):
    for dataset in data["datasets"]:
        if isinstance(dataset, dict) and "examples" in dataset:
            examples_list = dataset["examples"]
            if isinstance(examples_list, list):
                for example in examples_list:
                    if isinstance(example, dict):
                        # Parse example
                        try:
                            if isinstance(example.get("input"), str):
                                input_data = json.loads(example["input"])
                            else:
                                input_data = example.get("input", {})
                            
                            question1 = input_data.get("question1", "")
                            question2 = input_data.get("question2", "")
                            output = example.get("output", "0")
                            if isinstance(output, str):
                                label = int(output)
                            else:
                                label = int(output)
                            
                            if question1 and question2:
                                all_examples.append({
                                    "question1": question1,
                                    "question2": question2,
                                    "label": label,
                                    "metadata": example.get("metadata", {})
                                })
                        except Exception as e:
                            print(f"Failed to parse example: {e}")

print(f'Loaded {len(all_examples)} examples')
print(f'Duplicates: {sum(1 for e in all_examples if e["label"] == 1)}')
print(f'Non-duplicates: {sum(1 for e in all_examples if e["label"] == 0)}')

## Configuration

Set all tunable parameters here. For this demo, we use MINIMUM values to ensure quick execution.

**Tunable parameters:**
- `train_size`: Number of training examples (default: 5000, demo: 8)
- `val_size`: Number of validation examples (default: 1000, demo: 4)
- `test_size`: Number of test examples (default: 1000, demo: 8)
- `random_seed`: Random seed for reproducibility (default: 42)
- `dtw_radius`: Radius for fastdtw computation (default: 1)

**Note**: The original experiment used 5,000 train, 1,000 validation, and 4,000 test examples. This demo uses 8/4/8 for fast execution.

In [ ]:
# Configuration - using ABSOLUTE MINIMUM values for demo
TRAIN_SIZE = 8   # Minimum: need at least some training data for frequency dict
VAL_SIZE = 4     # Minimum: need validation data for threshold tuning
TEST_SIZE = 8    # Minimum: need test data for evaluation
RANDOM_SEED = 42
DTW_RADIUS = 1

print('Configuration set:')
print(f'  Train size: {TRAIN_SIZE}')
print(f'  Val size: {VAL_SIZE}')
print(f'  Test size: {TEST_SIZE}')
print(f'  Random seed: {RANDOM_SEED}')

## Create Train/Val/Test Splits

Split the data into stratified train, validation, and test sets.
Stratification ensures we maintain the same class balance across splits.

In [ ]:
# Create stratified splits
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Separate by label for stratification
duplicates = [e for e in all_examples if e["label"] == 1]
non_duplicates = [e for e in all_examples if e["label"] == 0]

print(f'Total duplicates: {len(duplicates)}')
print(f'Total non-duplicates: {len(non_duplicates)}')

# Calculate sizes per class (maintain balance)
total_size = TRAIN_SIZE + VAL_SIZE + TEST_SIZE
train_dup = min(len(duplicates) * TRAIN_SIZE // total_size, TRAIN_SIZE // 2) if len(duplicates) > 0 else 0
train_non = min(len(non_duplicates) * TRAIN_SIZE // total_size, TRAIN_SIZE // 2) if len(non_duplicates) > 0 else 0

val_dup = min(len(duplicates) - train_dup, VAL_SIZE // 2) if len(duplicates) > 0 else 0
val_non = min(len(non_duplicates) - train_non, VAL_SIZE // 2) if len(non_duplicates) > 0 else 0

test_dup = len(duplicates) - train_dup - val_dup
test_non = len(non_duplicates) - train_non - val_non

# Shuffle and split
random.shuffle(duplicates)
random.shuffle(non_duplicates)

train_data = duplicates[:train_dup] + non_duplicates[:train_non]
val_data = duplicates[train_dup:train_dup + val_dup] + non_duplicates[train_non:train_non + val_non]
test_data = duplicates[train_dup + val_dup:] + non_duplicates[train_non + val_non:]

random.shuffle(train_data)
random.shuffle(val_data)
random.shuffle(test_data)

print(f'\nSplit sizes:')
print(f'  Train: {len(train_data)} ({sum(e["label"] for e in train_data)} duplicates)')
print(f'  Val: {len(val_data)} ({sum(e["label"] for e in val_data)} duplicates)')
print(f'  Test: {len(test_data)} ({sum(e["label"] for e in test_data)} duplicates)')

## Reading Vector Construction

The RVP method constructs reading vectors from text using three cognitive load indicators:
1. **Word length** (normalized): Longer words may indicate more complex concepts
2. **Word frequency** (log scale): Common words vs. rare words
3. **POS tags** (integer encoded): Part-of-speech tags indicate grammatical structure

These features are combined into a reading vector for each token in the sequence.

In [ ]:
# Reading Vector Constructor class (from original method.py)
@dataclass
class MethodResult:
    """Results from a single method."""
    method_name: str
    precision: float
    recall: float
    f1: float
    roc_auc: float
    pr_auc: float
    avg_time_per_pair: float
    distances: List[float]
    predictions: List[int]

class ReadingVectorConstructor:
    """Constructs Reading Vectors from text using cognitive load indicators."""
    
    def __init__(self, freq_dict: Optional[Dict[str, float]] = None):
        self.freq_dict = freq_dict or {}
        self.pos_tag_map = self._create_pos_map()
    
    def _create_pos_map(self) -> Dict[str, int]:
        """Create mapping from POS tags to integer codes."""
        pos_tags = [
            'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS',
            'LS', 'MD', 'NN', 'NNS', 'NNP', 'NNPS', 'PDT', 'POS',
            'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO',
            'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT',
            'WP', 'WP$', 'WRB', 'PUNCT', 'UNK'
        ]
        return {tag: i for i, tag in enumerate(pos_tags)}
    
    def construct_reading_vector(self, text: str) -> np.ndarray:
        """Construct reading vector for a text."""
        if not text or not isinstance(text, str):
            return np.zeros((1, 3))
        
        # Tokenize
        tokens = word_tokenize(text.lower())
        if not tokens:
            return np.zeros((1, 3))
        
        # Get POS tags
        pos_tags = pos_tag(tokens)
        
        # Construct features
        features = []
        for token, (_, pos) in zip(tokens, pos_tags):
            # Feature 1: Word length (normalized by max 20)
            word_length = min(len(token) / 20.0, 1.0)
            
            # Feature 2: Word frequency (log scale, default for OOV)
            word_freq = self.freq_dict.get(token, self.freq_dict.get('UNK', 0.0))
            
            # Feature 3: POS code
            pos_code = self.pos_tag_map.get(pos, self.pos_tag_map['UNK']) / len(self.pos_tag_map)
            
            features.append([word_length, word_freq, pos_code])
        
        return np.array(features)
    
    def build_frequency_dict(self, examples: List[Dict]):
        """Build frequency dictionary from training examples."""
        print("Building frequency dictionary from training data...")
        word_counts = Counter()
        
        for example in examples:
            for question in [example["question1"], example["question2"]]:
                tokens = word_tokenize(question.lower())
                word_counts.update(tokens)
        
        # Convert to log frequency
        total_words = sum(word_counts.values())
        self.freq_dict = {'UNK': 0.0}
        
        for word, count in word_counts.items():
            self.freq_dict[word] = np.log1p(count) / np.log1p(total_words) if total_words > 0 else 0.0
        
        print(f"Frequency dictionary built with {len(self.freq_dict)} words")

print('ReadingVectorConstructor class defined!')

## DTW Comparator

Dynamic Time Warping (DTW) compares sequences of reading vectors to compute a distance.
DTW can handle sequences of different lengths by finding an optimal alignment.

We use `fastdtw` for faster computation with a radius parameter.

In [ ]:
# DTW Comparator class (from original method.py)
class DTWComparator:
    """Compares reading vectors using Dynamic Time Warping."""
    
    def __init__(self, radius: int = 1):
        self.radius = radius
    
    def compute_distance(self, rv1: np.ndarray, rv2: np.ndarray) -> float:
        """Compute DTW distance between two reading vectors."""
        if rv1.shape[0] == 0 or rv2.shape[0] == 0:
            return 1.0
        
        try:
            distance, _ = fastdtw(rv1, rv2, dist=euclidean, radius=self.radius)
            # Normalize by sequence lengths
            normalized_distance = distance / (rv1.shape[0] + rv2.shape[0])
            return normalized_distance
        except Exception as e:
            print(f"DTW computation failed: {e}")
            # Fallback to Euclidean distance on padded sequences
            return self._fallback_distance(rv1, rv2)
    
    def _fallback_distance(self, rv1: np.ndarray, rv2: np.ndarray) -> float:
        """Fallback distance computation."""
        min_len = min(rv1.shape[0], rv2.shape[0])
        max_len = max(rv1.shape[0], rv2.shape[0])
        
        if min_len == 0:
            return 1.0
        
        # Pad shorter sequence
        if rv1.shape[0] < rv2.shape[0]:
            rv1 = np.pad(rv1, ((0, rv2.shape[0] - rv1.shape[0]), (0, 0)), mode='edge')
        else:
            rv2 = np.pad(rv2, ((0, rv1.shape[0] - rv2.shape[0]), (0, 0)), mode='edge')
        
        return np.linalg.norm(rv1 - rv2) / max_len

print('DTWComparator class defined!')

## Baseline Methods

We compare RVP against three baseline methods:
1. **Jaccard 3-gram**: Character 3-gram Jaccard similarity
2. **MinHash LSH**: MinHash Locality-Sensitive Hashing similarity
3. **Sentence-BERT**: Cosine similarity of sentence embeddings from BERT

In [ ]:
# Baseline Methods class (from original method.py)
class BaselineMethods:
    """Implements baseline methods for comparison."""
    
    def __init__(self):
        self.sbert_model = None
    
    def jaccard_3gram(self, text1: str, text2: str) -> float:
        """Compute Jaccard similarity of character 3-grams."""
        def get_3grams(text):
            text = text.lower().replace(' ', '_')
            return set(text[i:i+3] for i in range(len(text) - 2))
        
        grams1 = get_3grams(text1)
        grams2 = get_3grams(text2)
        
        if not grams1 or not grams2:
            return 0.0
        
        intersection = len(grams1 & grams2)
        union = len(grams1 | grams2)
        
        similarity = intersection / union if union > 0 else 0.0
        return similarity  # Return similarity (higher = more likely duplicate)
    
    def minhash_lsh(self, text1: str, text2: str, num_perm: int = 128) -> float:
        """Compute MinHash LSH similarity."""
        def get_minhash(text, num_perm):
            m = MinHash(num_perm=num_perm)
            for word in text.lower().split():
                m.update(word.encode('utf8'))
            return m
        
        m1 = get_minhash(text1, num_perm)
        m2 = get_minhash(text2, num_perm)
        
        return m1.jaccard(m2)  # Return similarity (higher = more likely duplicate)
    
    def sentence_bert(self, text1: str, text2: str) -> float:
        """Compute Sentence-BERT cosine similarity."""
        if self.sbert_model is None:
            print("Loading Sentence-BERT model...")
            self.sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        try:
            embeddings = self.sbert_model.encode([text1, text2], show_progress_bar=False)
            similarity = np.dot(embeddings[0], embeddings[1]) / (
                np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1])
            )
            return similarity  # Return similarity (higher = more likely duplicate)
        except Exception as e:
            print(f"SBERT computation failed: {e}")
            return 0.5  # Default similarity

print('BaselineMethods class defined!')
baselines = BaselineMethods()  # Initialize for later use

## Evaluate RVP Method

Now we evaluate the RVP method:
1. Build frequency dictionary from training data
2. Compute distances on validation set to find optimal threshold
3. Compute distances on test set and evaluate metrics

In [ ]:
# Initialize RVP components
rv_constructor = ReadingVectorConstructor()
dtw_comparator = DTWComparator(radius=DTW_RADIUS)

# Build frequency dictionary from training data
rv_constructor.build_frequency_dict(train_data)

print('\nStep 1: Compute RVP distances on validation set...')
val_distances = []
for example in tqdm(val_data, desc="RVP val"):
    rv1 = rv_constructor.construct_reading_vector(example["question1"])
    rv2 = rv_constructor.construct_reading_vector(example["question2"])
    dist = dtw_comparator.compute_distance(rv1, rv2)
    val_distances.append(dist)

val_labels = [e["label"] for e in val_data]

# Find optimal threshold (lower distance = more likely duplicate for RVP)
print('\nStep 2: Find optimal threshold...')
candidates = np.linspace(min(val_distances), max(val_distances), 50)  # Reduced from 100 for speed
best_f1 = 0
best_threshold = candidates[0]

for threshold in candidates:
    predictions = [1 if d <= threshold else 0 for d in val_distances]
    f1 = f1_score(val_labels, predictions, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f'Optimal threshold: {best_threshold:.4f} (F1={best_f1:.4f})')

print('\nStep 3: Compute RVP distances on test set...')
test_distances = []
for example in tqdm(test_data, desc="RVP test"):
    rv1 = rv_constructor.construct_reading_vector(example["question1"])
    rv2 = rv_constructor.construct_reading_vector(example["question2"])
    dist = dtw_comparator.compute_distance(rv1, rv2)
    test_distances.append(dist)

test_labels = [e["label"] for e in test_data]
test_predictions = [1 if d <= best_threshold else 0 for d in test_distances]

# Compute metrics for RVP
rvp_precision = precision_score(test_labels, test_predictions, zero_division=0)
rvp_recall = recall_score(test_labels, test_predictions, zero_division=0)
rvp_f1 = f1_score(test_labels, test_predictions, zero_division=0)

# ROC-AUC (lower distance = higher score = more likely duplicate)
rvp_scores = -np.array(test_distances)  # Negate so higher = more likely duplicate
rvp_roc_auc = roc_auc_score(test_labels, rvp_scores) if len(set(test_labels)) > 1 else 0.5
rvp_pr_auc = average_precision_score(test_labels, rvp_scores) if len(set(test_labels)) > 1 else 0.0

print(f'\nRVP Results:')
print(f'  Precision: {rvp_precision:.4f}')
print(f'  Recall: {rvp_recall:.4f}')
print(f'  F1-Score: {rvp_f1:.4f}')
print(f'  ROC-AUC: {rvp_roc_auc:.4f}')
print(f'  PR-AUC: {rvp_pr_auc:.4f}')

## Evaluate Baseline Methods

Now we evaluate the three baseline methods on the test set.

In [ ]:
# Evaluate Jaccard baseline
print('Evaluating Jaccard 3-gram baseline...')
jaccard_similarities = []
for example in tqdm(test_data, desc="Jaccard"):
    sim = baselines.jaccard_3gram(example["question1"], example["question2"])
    jaccard_similarities.append(sim)

# Find optimal threshold for Jaccard (higher similarity = more likely duplicate)
candidates = np.linspace(min(jaccard_similarities), max(jaccard_similarities), 50)
best_f1 = 0
best_threshold = candidates[0]

for threshold in candidates:
    predictions = [1 if s >= threshold else 0 for s in jaccard_similarities]
    f1 = f1_score(test_labels, predictions, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

jaccard_predictions = [1 if s >= best_threshold else 0 for s in jaccard_similarities]
jaccard_precision = precision_score(test_labels, jaccard_predictions, zero_division=0)
jaccard_recall = recall_score(test_labels, jaccard_predictions, zero_division=0)
jaccard_f1 = f1_score(test_labels, jaccard_predictions, zero_division=0)
jaccard_roc_auc = roc_auc_score(test_labels, jaccard_similarities) if len(set(test_labels)) > 1 else 0.5
jaccard_pr_auc = average_precision_score(test_labels, jaccard_similarities) if len(set(test_labels)) > 1 else 0.0

print(f'\nJaccard Results:')
print(f'  Precision: {jaccard_precision:.4f}')
print(f'  Recall: {jaccard_recall:.4f}')
print(f'  F1-Score: {jaccard_f1:.4f}')
print(f'  ROC-AUC: {jaccard_roc_auc:.4f}')
print(f'  PR-AUC: {jaccard_pr_auc:.4f}')

In [ ]:
# Evaluate MinHash LSH baseline
print('Evaluating MinHash LSH baseline...')
minhash_similarities = []
for example in tqdm(test_data, desc="MinHash"):
    sim = baselines.minhash_lsh(example["question1"], example["question2"])
    minhash_similarities.append(sim)

# Find optimal threshold
candidates = np.linspace(min(minhash_similarities), max(minhash_similarities), 50)
best_f1 = 0
best_threshold = candidates[0]

for threshold in candidates:
    predictions = [1 if s >= threshold else 0 for s in minhash_similarities]
    f1 = f1_score(test_labels, predictions, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

minhash_predictions = [1 if s >= best_threshold else 0 for s in minhash_similarities]
minhash_precision = precision_score(test_labels, minhash_predictions, zero_division=0)
minhash_recall = recall_score(test_labels, minhash_predictions, zero_division=0)
minhash_f1 = f1_score(test_labels, minhash_predictions, zero_division=0)
minhash_roc_auc = roc_auc_score(test_labels, minhash_similarities) if len(set(test_labels)) > 1 else 0.5
minhash_pr_auc = average_precision_score(test_labels, minhash_similarities) if len(set(test_labels)) > 1 else 0.0

print(f'\nMinHash LSH Results:')
print(f'  Precision: {minhash_precision:.4f}')
print(f'  Recall: {minhash_recall:.4f}')
print(f'  F1-Score: {minhash_f1:.4f}')
print(f'  ROC-AUC: {minhash_roc_auc:.4f}')
print(f'  PR-AUC: {minhash_pr_auc:.4f}')

In [ ]:
# Evaluate Sentence-BERT baseline
print('Evaluating Sentence-BERT baseline...')
sbert_similarities = []
for example in tqdm(test_data, desc="SBERT"):
    sim = baselines.sentence_bert(example["question1"], example["question2"])
    sbert_similarities.append(sim)

# Find optimal threshold
candidates = np.linspace(min(sbert_similarities), max(sbert_similarities), 50)
best_f1 = 0
best_threshold = candidates[0]

for threshold in candidates:
    predictions = [1 if s >= threshold else 0 for s in sbert_similarities]
    f1 = f1_score(test_labels, predictions, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

sbert_predictions = [1 if s >= best_threshold else 0 for s in sbert_similarities]
sbert_precision = precision_score(test_labels, sbert_predictions, zero_division=0)
sbert_recall = recall_score(test_labels, sbert_predictions, zero_division=0)
sbert_f1 = f1_score(test_labels, sbert_predictions, zero_division=0)
sbert_roc_auc = roc_auc_score(test_labels, sbert_similarities) if len(set(test_labels)) > 1 else 0.5
sbert_pr_auc = average_precision_score(test_labels, sbert_similarities) if len(set(test_labels)) > 1 else 0.0

print(f'\nSentence-BERT Results:')
print(f'  Precision: {sbert_precision:.4f}')
print(f'  Recall: {sbert_recall:.4f}')
print(f'  F1-Score: {sbert_f1:.4f}')
print(f'  ROC-AUC: {sbert_roc_auc:.4f}')
print(f'  PR-AUC: {sbert_pr_auc:.4f}')

## Results Visualization

Compare all methods and visualize the results.

In [ ]:
# Compile results into a table
results = {
    'RVP': {
        'Precision': rvp_precision,
        'Recall': rvp_recall,
        'F1-Score': rvp_f1,
        'ROC-AUC': rvp_roc_auc,
        'PR-AUC': rvp_pr_auc
    },
    'Jaccard': {
        'Precision': jaccard_precision,
        'Recall': jaccard_recall,
        'F1-Score': jaccard_f1,
        'ROC-AUC': jaccard_roc_auc,
        'PR-AUC': jaccard_pr_auc
    },
    'MinHash': {
        'Precision': minhash_precision,
        'Recall': minhash_recall,
        'F1-Score': minhash_f1,
        'ROC-AUC': minhash_roc_auc,
        'PR-AUC': minhash_pr_auc
    },
    'SBERT': {
        'Precision': sbert_precision,
        'Recall': sbert_recall,
        'F1-Score': sbert_f1,
        'ROC-AUC': sbert_roc_auc,
        'PR-AUC': sbert_pr_auc
    }
}

# Print results table
print('=' * 80)
print('RESULTS SUMMARY')
print('=' * 80)
header = f"{'Method':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'ROC-AUC':<12} {'PR-AUC':<12}"
print(header)
print('-' * 80)

for method, metrics in results.items():
    row = f"{method:<12} {metrics['Precision']:<12.4f} {metrics['Recall']:<12.4f} {metrics['F1-Score']:<12.4f} {metrics['ROC-AUC']:<12.4f} {metrics['PR-AUC']:<12.4f}"
    print(row)

print('=' * 80)

In [ ]:
# Visualize results with a bar chart
methods = list(results.keys())
metrics = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('RVP vs Baselines: Performance Comparison', fontsize=16)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    values = [results[m][metric] for m in methods]
    
    bars = ax.bar(methods, values, color=colors)
    ax.set_title(f'{metric}', fontsize=14)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_ylim([0, 1.0])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{val:.3f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

# Print conclusion
print('\n' + '=' * 80)
print('CONCLUSION')
print('=' * 80)
print('This demo shows the RVP method for near-duplicate detection.')
print('Key observations:')
print('  1. RVP uses cognitive load indicators (word length, frequency, POS)')
print('  2. RVP is compared against Jaccard, MinHash, and Sentence-BERT baselines')
print('  3. With the mini demo dataset (20 examples), results may vary due to small sample size')
print('  4. For production use, run with the full QQP dataset (5000+ examples)')
print('=' * 80)